In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

"""
This script evaluates cross-sectional prediction accuracy for t + horizon returns.

It produces two output files:

1. prediction_daily_ic_rankic.csv
   - Daily cross-sectional metrics:
       - rank_ic : Rank Information Coefficient (Spearman) between predictions
                   and realized forward returns
       - ic      : Pearson Information Coefficient between predictions
                   and realized forward returns

2. prediction_summary_metrics.csv
   - Aggregated performance metrics (single-row summary):
       - mean_rank_ic : Time-series average of daily RankIC
       - mean_ic      : Time-series average of daily IC
       - mse          : Mean Squared Error over all samples
       - n_days       : Number of trading days evaluated
       - n_samples    : Total number of valid samples
       - horizon      : Prediction horizon (t + horizon)
"""

# ===================== Basic Configuration (Aligned with Linear Regression / DL models) =====================

# Root directory for model outputs
OUTPUT_DIR = "/content/drive/My Drive/Deep Learning Model/"
PREDICTION_FILE = OUTPUT_DIR + "predictions_1.csv"
ANALYSIS_OUTPUT_DIR = OUTPUT_DIR +"analysis/"

DATE_COL = "dt"
ID_COL = "corp_tkr"
HORIZON = 1

PRED_COL = f"pred_{HORIZON}"
ACTUAL_COL = f"ret_fwd_{HORIZON}"

MIN_STOCKS_PER_DAY = 30



def load_and_prepare_base_df() -> pd.DataFrame:
    """
    Load the prediction file and perform basic cleaning and preparation.

    Processing steps:
    1. Read predictions.csv
    2. Convert the date column to datetime
    3. If split_id exists, keep only split_id > 0 (out-of-sample data)
    4. Validate required columns
    5. Drop rows with missing predictions or realized returns
    6. Compute prediction deviation (actual - predicted)

    Returns
    -------
    pd.DataFrame
        Cleaned prediction DataFrame ready for evaluation.
    """
    df = pd.read_csv(PREDICTION_FILE)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])

    print(f"✅ read predictions file successfully: {PREDICTION_FILE}")
    print(f"   total {len(df):,} rows")


    if "split_id" in df.columns:
        df = df[df["split_id"] > 0].copy()
    needed_cols = [DATE_COL, ID_COL, PRED_COL, ACTUAL_COL]
    missing = [c for c in needed_cols if c not in df.columns]
    if missing:
        raise ValueError(f"predictions.csv missing: {missing}")

    df = df.dropna(subset=[PRED_COL, ACTUAL_COL])
    df["deviation"] = df[ACTUAL_COL] - df[PRED_COL]

    print(f"   effective rows: {len(df):,}")
    return df


def compute_prediction_accuracy_metrics(df: pd.DataFrame):
    """
    Compute cross-sectional prediction accuracy metrics.

    Metrics computed:
    ------------------
    1. Daily metrics (by trading date):
       - RankIC : Spearman rank correlation between predictions and realized returns
       - IC     : Pearson correlation between predictions and realized returns

    2. Summary metrics:
       - mean_rank_ic : Average daily RankIC
       - mean_ic      : Average daily IC
       - mse          : Mean Squared Error over all samples

    Parameters
    ----------
    df : pd.DataFrame
        Cleaned prediction DataFrame.

    Returns
    -------
    daily_metrics : pd.DataFrame
        Daily RankIC and IC indexed by date.
    summary_df : pd.DataFrame
        Single-row DataFrame with aggregated performance metrics.
    """

    def _daily_metrics(group: pd.DataFrame) -> pd.Series:
        """
        Compute cross-sectional metrics for a single trading day.

        Days with insufficient cross-sectional coverage or no variation
        in predictions / realized returns are excluded (return NaN).
        """
        g = group[[PRED_COL, ACTUAL_COL]].dropna()
        if len(g) < MIN_STOCKS_PER_DAY:
            return pd.Series({"rank_ic": np.nan, "ic": np.nan})
        if g[PRED_COL].nunique() < 2 or g[ACTUAL_COL].nunique() < 2:
            return pd.Series({"rank_ic": np.nan, "ic": np.nan})

        rank_ic = g[PRED_COL].rank().corr(g[ACTUAL_COL].rank())
        ic = g[PRED_COL].corr(g[ACTUAL_COL])
        return pd.Series({"rank_ic": rank_ic, "ic": ic})

    daily_metrics = (
        df.groupby(DATE_COL)
          .apply(_daily_metrics)
          .sort_index()
    )

    summary = {
        "mean_rank_ic": daily_metrics["rank_ic"].mean(skipna=True),
        "mean_ic": daily_metrics["ic"].mean(skipna=True),
        "mse": float(np.mean((df[ACTUAL_COL] - df[PRED_COL]) ** 2)),
        "n_days": int(daily_metrics.shape[0]),
        "n_samples": int(len(df)),
        "horizon": int(HORIZON),
    }

    summary_df = pd.DataFrame([summary])
    return daily_metrics, summary_df


def main():
    """
    Main execution workflow:
    1. Load and clean prediction data
    2. Compute daily IC / RankIC and aggregated metrics
    3. Save results to CSV files
    """
    df = load_and_prepare_base_df()

    print("📈 Computing cross-sectional prediction accuracy metrics (t + horizon)...")
    daily_pred_metrics, pred_summary = compute_prediction_accuracy_metrics(df)

    print("✅ Prediction accuracy summary:")
    print(pred_summary)

    daily_pred_metrics.to_csv(
        ANALYSIS_OUTPUT_DIR + "prediction_daily_ic_rankic.csv"
    )
    pred_summary.to_csv(
        ANALYSIS_OUTPUT_DIR + "prediction_summary_metrics.csv",
        index=False
    )

    print("✅ All analysis results saved to:", ANALYSIS_OUTPUT_DIR)


if __name__ == "__main__":
    main()


In [ ]:

print("\n📊 Aligned average RankIC / IC by day-in-test-window:")
df = load_and_prepare_base_df()
daily_pred_metrics, pred_summary = compute_prediction_accuracy_metrics(df)

daily = daily_pred_metrics.copy()
daily = daily.sort_index()
TEST_WINDOW= 10
window = TEST_WINDOW  # e.g. 10
daily = daily.reset_index()

daily["day_in_test"] = daily.index % window
aligned_stats = (
    daily
    .groupby("day_in_test")[["rank_ic", "ic"]]
    .mean()
    .reset_index()
)

aligned_stats["day_in_test"] = aligned_stats["day_in_test"] + 1
aligned_stats = aligned_stats.rename(columns={
    "rank_ic": "mean_rank_ic",
    "ic": "mean_ic",
})

# print
for _, row in aligned_stats.iterrows():
    print(
        f"Day {int(row['day_in_test']):2d} | "
        f"mean RankIC = {row['mean_rank_ic']:.4f}, "
        f"mean IC = {row['mean_ic']:.4f}"
    )



aligned_output_path = ANALYSIS_OUTPUT_DIR + "prediction_aligned_rankic_ic.csv"
aligned_stats.to_csv(aligned_output_path, index=False)

print(f"\n✅ Aligned RankIC / IC saved to: {aligned_output_path}")
